# 02 – Evaluation design and baseline

Splitting the data, building the preprocessing pipeline and scoring a simple baseline. Decisions here are based on the findings in `01_data_audit.ipynb`.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/ai4i2020.csv")

target = "Machine failure"
drop_cols = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]

X = df.drop(columns=drop_cols + [target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

X shape: (10000, 6)
y shape: (10000,)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


**What I did:**
- Dropped `UDI` and `Product ID` because they are identifiers (leakage register, risk 2).
- Dropped `TWF`, `HDF`, `PWF`, `OSF` and `RNF` because they reveal the target (leakage register, risk 1).
- `X` has the 6 features (`Type` and the five measurements) and `y` is `Machine failure`.

## 2. Evaluation protocol
Written before running any models (Week 4).

| # | Point | Decision |
|---|---|---|
| 1 | Prediction context | One row is one product being machined. Using the current sensor readings and product type, predict whether the machine will fail, so it can be flagged for a maintenance check. Results apply only to this simulated milling process. |
| 2 | Test strategy | Stratified random 80/20 hold-out. The rows are in time order (tool wear increases in 98.8% of consecutive rows and air temperature drifts slowly), but each row is a different product and failure depends mainly on the row's own readings (failure is barely related to the previous row, autocorrelation 0.05). A random split is therefore reasonable, but it may be slightly optimistic for future data, so I will also check a time-ordered split in Milestone 3. Stratification is needed because only 3.4% of rows are failures. |
| 3 | Cross-validation | `StratifiedKFold`, 5 folds, shuffled, seed 42, on the training set only. 5 folds keep about 54 failures in each validation fold; 10 folds would leave only about 27. |
| 4 | Baselines | Dummy (always predicts "no failure") and Logistic Regression as a simple learned baseline (Week 4). |
| 5 | Metrics | **Primary: F2**, which weights recall more than precision because missed failures are costlier. Secondary: PR-AUC (average precision), precision, recall and F1 for the failure class. Accuracy is reported only for context. Diagnostics in Milestone 3: confusion matrix and precision–recall curve. |
| 6 | Error costs | False negative (missed failure): unplanned downtime, repair cost and possible safety risk, so this is the most costly error. False positive (false alarm): an unnecessary inspection, which costs time but is far less serious. |
| 7 | Leakage risks | (a) Fitting scaling or encoding outside cross-validation, prevented by a `Pipeline`. (b) Tuning hyperparameters, features or the threshold on the test set, prevented by using CV only. (c) Using failure-type or ID columns as features, prevented by dropping them in Section 1. |
| 8 | Test-set rule | The test set is opened once, in Milestone 3, after the models, hyperparameters, features and threshold have all been fixed using cross-validation. The test results are reported as they are, and nothing is changed afterwards. |

**Change from my proposal:** the proposal listed precision, recall and F1. After Week 4 I chose F2 as the primary metric because it reflects that missed failures cost more than false alarms. Precision, recall and F1 are kept as secondary metrics.

## 3. Configuration and seed policy
All randomness uses one seed, and the package versions are printed so results can be reproduced (Week 4).

In [5]:
import sys
import sklearn

SEED = 42
TEST_SIZE = 0.2
N_FOLDS = 5

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.11
pandas: 3.0.6
scikit-learn: 1.9.1


## 4. Train/test split

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Failure rate overall:", round(y.mean(), 4))
print("Failure rate train:  ", round(y_train.mean(), 4))
print("Failure rate test:   ", round(y_test.mean(), 4))

Train: (8000, 6) Test: (2000, 6)
Failure rate overall: 0.0339
Failure rate train:   0.0339
Failure rate test:    0.034


**Why this design:**
- **80/20 split:** 8,000 rows to train on and 2,000 held back to test the final models.
- **Stratified (`stratify=y`):** only 3.4% of rows are failures, so a plain random split could give the test set noticeably more or fewer failures. The output above shows both sets keep about 3.4%.
- **Random rather than time-ordered:** the rows are in time order, but failure depends mainly on each row's own readings (see protocol point 2 and the time-order check in `01_data_audit.ipynb`). A time-ordered split will be checked in Milestone 3.
- **Fixed seed (`SEED`):** the split is the same every time the notebook runs.
- **The test set is locked away:** it follows the test-set rule in protocol point 8. All development uses cross-validation on the training set.